# LizyML Tutorial: Regression with LightGBM

This notebook demonstrates the full LizyML workflow on the **Diamonds** dataset:

1. Data preparation (download from URL, includes categorical columns)
2. Config setup
3. Model fit (5-fold CV with early stopping)
4. Evaluate — metrics table + learning curve
5. Residual distribution
6. Feature importance — Split (plot + table)
7. Feature importance — Gain (plot + table)
8. Feature importance — SHAP (plot + table)

**Dataset**: Diamonds (~54,000 rows)  
**Target**: `price` (USD)  
**Categorical features**: `cut`, `color`, `clarity`

## 1. Setup

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from lizyml import Model

## 2. Data Preparation

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"

df = pd.read_csv(URL)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
# Categorical columns that LizyML will auto-detect and encode
cat_cols = ["cut", "color", "clarity"]
for col in cat_cols:
    print(f"{col}: {sorted(df[col].unique())}")

In [ ]:
df.describe()

## 3. Config

In [ ]:
config = {
    "config_version": 1,
    "task": "regression",
    "data": {
        "target": "price",
    },
    "split": {
        "method": "kfold",
        "n_splits": 5,
        "random_state": 42,
        "shuffle": True,
    },
    "model": {
        "name": "lgbm",
        "params": {
            "n_estimators": 1000,
            "learning_rate": 0.05,
            "num_leaves": 63,
            "min_child_samples": 20,
            "verbose": -1,
        },
    },
    "features": {
        "auto_categorical": True,  # auto-detects cut/color/clarity
    },
    "training": {
        "seed": 42,
        "early_stopping": {
            "enabled": True,
            "rounds": 50,
            "inner_valid": {"method": "holdout", "ratio": 0.1},
        },
    },
    "evaluation": {
        "metrics": ["mae", "rmse", "r2", "mape", "huber"],
    },
}

## 4. Model Fit

In [ ]:
model = Model(config)
fit_result = model.fit(data=df)
print("Fit complete.")

## 5. Evaluate

### 5.1 Metrics Table

In [ ]:
model.evaluate_table().round(4)

### 5.2 Learning Curve

In [ ]:
model.plot_learning_curve()

## 6. Residual Distribution

In [ ]:
model.residuals_plot()

In [ ]:
y_true = df["price"].to_numpy(dtype=float)
oof_pred = fit_result.oof_pred

lim = [min(y_true.min(), oof_pred.min()), max(y_true.max(), oof_pred.max())]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=oof_pred, y=y_true,
    mode="markers",
    marker=dict(size=2, opacity=0.3),
    name="OOF predictions",
))
fig.add_trace(go.Scatter(
    x=lim, y=lim,
    mode="lines",
    line=dict(color="red", dash="dash"),
    name="Perfect prediction",
))
fig.update_layout(
    title="Actual vs Predicted (OOF)",
    xaxis_title="Predicted price",
    yaxis_title="Actual price",
    height=500,
)
fig

## 7. Feature Importance: Split

In [ ]:
model.importance_plot(kind="split")

In [ ]:
imp_split = (
    pd.Series(model.importance(kind="split"), name="importance_split")
    .sort_values(ascending=False)
    .to_frame()
)
imp_split

## 8. Feature Importance: Gain

In [ ]:
model.importance_plot(kind="gain")

In [ ]:
imp_gain = (
    pd.Series(model.importance(kind="gain"), name="importance_gain")
    .sort_values(ascending=False)
    .to_frame()
)
imp_gain

## 9. Feature Importance: SHAP

`model.importance(kind="shap")` computes fold-averaged SHAP importance internally:
for each CV fold, TreeSHAP is run on the validation subset and mean(|SHAP|) is computed per feature.
The fold-level scores are then averaged across all folds.

SHAP values always have shape `(n_samples, n_features)` regardless of task.

In [ ]:
shap_imp = model.importance(kind="shap")
pd.Series(shap_imp, name="mean_abs_shap").sort_values(ascending=False).to_frame()

In [ ]:
model.importance_plot(kind="shap")